# 02 — Exploratory Data Analysis

Visualize and summarize historical deforestation patterns in Brazil's 27 regions (2001–2022).


In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.express as px
from config import *
from src.data_loader import load_all_datasets
from src.preprocessing import (
    preprocess_subnational_tree_loss, preprocess_carbon_data, merge_carbon_with_loss
)
from src.eda import summary_statistics, compute_regional_trends, correlation_analysis
from src.visualization import *


## Load Preprocessed Data


In [2]:
datasets = load_all_datasets()
processed = preprocess_subnational_tree_loss(
    datasets['subnational_loss'], threshold=30, country='Brazil'
)
carbon_long = preprocess_carbon_data(
    datasets['subnational_carbon'], threshold=30, country='Brazil'
)
processed = merge_carbon_with_loss(processed, carbon_long)
print(f'Processed dataset: {processed.shape}')
processed.head()


2026-11-21 00:15:10,083 - deforestation_ml.data_loader - INFO - Loaded Country Tree Cover Loss: 1888 rows × 28 columns from Country tree cover loss.csv


2026-11-21 00:15:10,103 - deforestation_ml.data_loader - INFO - Loaded Subnational Tree Cover Loss: 28000 rows × 29 columns from Subnational 1 tree cover loss.csv


2026-11-21 00:15:10,108 - deforestation_ml.data_loader - INFO - Loaded Country Carbon Data: 1888 rows × 30 columns from Country carbon data.csv


2026-11-21 00:15:10,128 - deforestation_ml.data_loader - INFO - Loaded Subnational Carbon Data: 28000 rows × 31 columns from Subnational 1 carbon data.csv


2026-11-21 00:15:10,132 - deforestation_ml.preprocessing - INFO - Filtered to threshold=30: 3500 rows (from 28000)


2026-11-21 00:15:10,134 - deforestation_ml.preprocessing - INFO - Filtered to country='Brazil': 27 rows


... (2 outputs left out)

2026-11-21 00:15:10,152 - deforestation_ml.preprocessing - INFO - Filtered to threshold=30: 3500 rows (from 28000)


2026-11-21 00:15:10,165 - deforestation_ml.preprocessing - INFO - Merged carbon data: 594/594 rows matched (100.0%)


Processed dataset: (594, 12)


,country,region,total_area_ha,extent_2000_ha,extent_2010_ha,gain_2000_2020_ha,tc_loss_ha,year,loss_rate_pct,cumulative_loss_ha,cumulative_loss_pct,carbon_emissions_Mg_CO2e
0,Brazil,Acre,15273530,14379380,14079300,66320,67310,2001,0.468101,67310,0.468101,43237142.0
1,Brazil,Acre,15273530,14379380,14079300,66320,68439,2002,0.475952,135749,0.944053,48986317.0
2,Brazil,Acre,15273530,14379380,14079300,66320,37666,2003,0.261945,173415,1.205998,26082167.0
3,Brazil,Acre,15273530,14379380,14079300,66320,67190,2004,0.467266,240605,1.673264,48123664.0
4,Brazil,Acre,15273530,14379380,14079300,66320,169860,2005,1.181275,410465,2.854539,117949735.0


## Summary Statistics


In [3]:
stats = summary_statistics(processed)
print(f'Total forest loss: {stats["total_loss_ha"]:,.0f} ha')
print(f'Average annual loss: {stats["avg_annual_loss"]:,.0f} ha')
print(f'Peak year: {stats["peak_year"]} ({stats["peak_loss"]:,.0f} ha)')
print(f'Number of regions: {stats["n_regions"]}')
print(f'Top 5 regions: {", ".join(stats["top_5_regions"])}')
print(f'Regions with increasing trends: {len(stats["increasing_regions"])}')
print(f'Regions with decreasing trends: {len(stats["decreasing_regions"])}')


Total forest loss: 66,126,533 ha
Average annual loss: 3,005,752 ha
Peak year: 2016 (5,378,844 ha)
Number of regions: 27
Top 5 regions: Pará, Mato Grosso, Maranhão, Rondônia, Amazonas
Regions with increasing trends: 12
Regions with decreasing trends: 15


## Total Annual Tree-Cover Loss


In [4]:
plot_total_annual_loss(processed).show()


## Top 15 Regions by Total Loss


In [5]:
plot_regional_comparison(processed, top_n=15).show()


## Annual Trends — Top 10 Regions


In [6]:
top_regions = processed.groupby('region')['tc_loss_ha'].sum().nlargest(10).index
plot_data = processed[processed['region'].isin(top_regions)]
plot_annual_trend(plot_data, color_col='region',
                  title='Annual Tree-Cover Loss — Top 10 Regions').show()


## Heatmap: Region × Year


In [7]:
plot_heatmap(processed).show()


## Loss Rate Heatmap


In [8]:
plot_loss_rate_heatmap(processed).show()


## Cumulative Forest Loss


In [9]:
plot_cumulative_loss(processed, top_n=10).show()


## Regional Trend Analysis


In [10]:
trends = compute_regional_trends(processed)
trends = trends.sort_values('trend_slope', ascending=False)
print('Regions with INCREASING deforestation trends:')
print(trends[trends['trend_slope'] > 0][['region', 'trend_slope', 'total_loss']].to_string(index=False))
print()
print('Regions with DECREASING deforestation trends:')
print(trends[trends['trend_slope'] < 0][['region', 'trend_slope', 'total_loss']].to_string(index=False))


Regions with INCREASING deforestation trends:
           region  trend_slope  total_loss
         Maranhão      14621.4   5881361.0
         Amazonas      11851.3   4020611.0
             Pará       6450.6  16330101.0
            Piauí       2992.6   1445456.0
        Tocantins       2601.9   2159950.0
          Roraima       2479.4   1167491.0
             Acre       2061.8   1574809.0
Rio Grande do Sul       1485.0    794836.0
   Santa Catarina        635.5    956916.0
           Paraná        471.2   1355647.0
            Amapá         55.8    339893.0
   Rio de Janeiro         53.9     80645.0

Regions with DECREASING deforestation trends:
             region  trend_slope  total_loss
   Distrito Federal         -0.7      3271.0
     Espírito Santo       -122.3    387146.0
              Ceará       -191.2    335695.0
            Alagoas       -277.1     77463.0
              Bahia       -285.5   3638717.0
            Paraíba       -338.8    128896.0
Rio Grande do Norte       -350.3 

## Correlation Analysis


In [11]:
corr = correlation_analysis(processed)
print('Pairwise correlations:')
print(corr.round(3))


Pairwise correlations:
                          tc_loss_ha  loss_rate_pct  cumulative_loss_ha  \
tc_loss_ha                     1.000          0.402               0.761   
loss_rate_pct                  0.402          1.000               0.213   
cumulative_loss_ha             0.761          0.213               1.000   
carbon_emissions_Mg_CO2e       0.982          0.337               0.792   

                          carbon_emissions_Mg_CO2e  
tc_loss_ha                                   0.982  
loss_rate_pct                                0.337  
cumulative_loss_ha                           0.792  
carbon_emissions_Mg_CO2e                     1.000  


## Carbon Emissions Trends


In [12]:
plot_carbon_trends(processed, top_n=10).show()


## Annual Carbon vs Tree-Cover Loss


In [13]:
valid = processed[processed['carbon_emissions_Mg_CO2e'].notna()]
fig = px.scatter(
    valid, x='tc_loss_ha', y='carbon_emissions_Mg_CO2e',
    color='region', opacity=0.5,
    title='Tree-Cover Loss vs Carbon Emissions',
    labels={'tc_loss_ha': 'Loss (ha)', 'carbon_emissions_Mg_CO2e': 'Emissions (Mg CO₂e)'},
)
fig.show()
corr_val = valid[['tc_loss_ha', 'carbon_emissions_Mg_CO2e']].corr().iloc[0, 1]
print(f'Pearson r = {corr_val:.4f}')


Pearson r = 0.9816
